# Random Forest and LightGBM Implementation

This notebook implements the machine-learning part of the experimental described in the report. It reuses the selected M4 monthly series, formulates forecasting as supervised regression with lag-based features, tunes Random Forest and LightGBM separately for each series, forecasts the 18-month test horizon, and stores the final errors for comparison with ARIMA and the baselines.

**Report references:** Sections 3.1 (*Data*), 3.2 (*Methodology*), and 4 (*Results and Analysis*).

## 1. Libraries and reproducibility

Random Forest and LightGBM are evaluated as machine-learning forecasting models. This cell imports the data-loading, MLForecast, Optuna, Random Forest, LightGBM, metric, and parallel-execution tools required for that experiment. The fixed seed supports reproducibility.

**Report reference:** Section 3.2 (*Models*).

In [1]:
import os
import random
import optuna

from joblib import Parallel, delayed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasetsforecast.m4 import M4

from sklearn.ensemble import RandomForestRegressor

from window_ops.rolling import rolling_mean
from window_ops.expanding import expanding_mean

from mlforecast.auto import AutoMLForecast, AutoRandomForest, AutoLightGBM
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences

from lightgbm import LGBMRegressor

import utilsforecast.losses as ufl
from utilsforecast.evaluation import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

## 2. Load the M4 monthly data

The empirical analysis is based on monthly series from the M4 Competition dataset. This cell loads the same monthly subset used by the ARIMA.

**Report reference:** Section 3.1 (*Data*).

In [2]:
M4.download(directory="data", group="Monthly") 

df, *_ = M4.load(directory="../data", group="Monthly")

df.head()

,unique_id,ds,y
0,M1,1,8000.0
1,M1,2,8350.0
2,M1,3,8570.0
3,M1,4,7700.0
4,M1,5,7080.0


## 3. Reuse the selected series from the shared results table

The report selects 400 series using a stratified pseudo-random strategy based on CE. This notebook loads the `final_df` produced earlier and filters the M4 data to the same selected series, ensuring that Random Forest and LightGBM are evaluated on exactly the same instances as AutoARIMA and the baselines.

**Report reference:** Section 3.1.1 (*Stratified Pseudo-Random Selection Strategy*).

In [3]:
final_df = pd.read_csv("../data_selection_and_ARIMA_implementation/final_df.csv")

series_ids = final_df["unique_id"].unique().tolist()

df = df[df["unique_id"].isin(series_ids)].copy()

df["ds"] = df["ds"].astype(int)

df = (
    df.assign(
        uid_num=df["unique_id"].str[1:].astype(int)
    )
    .sort_values(["uid_num", "ds"])
    .drop(columns="uid_num")
    .reset_index(drop=True)
)

df.head()

,unique_id,ds,y
0,M80,1,4580.0
1,M80,2,4260.0
2,M80,3,4430.0
3,M80,4,3400.0
4,M80,5,3300.0


## 4. Inspect the shared results table

`final_df` is the central table where CE values, metadata, baseline errors, AutoARIMA errors, and machine-learning errors are accumulated. Keeping all results in this table supports the model comparisons presented later in the report.

**Report references:** Sections 3.2 (*Methodology*) and 4 (*Results and Analysis*).

In [4]:
final_df.head()

,unique_id,size,complexity,transformations,n_transformations,final_status,nmae_AutoARIMA,nrmse_AutoARIMA,smape_AutoARIMA,nmae_naive,...,smape_naive,nmae_seasonal_naive,nrmse_seasonal_naive,smape_seasonal_naive,%nmae_arima_vs_snaive,%nrmse_arima_vs_snaive,%smape_arima_vs_snaive,%nmae_arima_vs_naive,%nrmse_arima_vs_naive,%smape_arima_vs_naive
0,M80,585,22292.093,[],0,stationary,0.119518,0.154520,0.060977,0.323598,...,0.143624,0.143448,0.169693,0.071112,16.682007,8.941047,14.252458,63.065906,56.191667,57.544181
1,M160,91,273.198,['difference'],1,stationary,0.153033,0.178445,0.080407,0.148942,...,0.078029,0.151324,0.176683,0.079474,-1.129450,-0.997299,-1.173563,-2.746526,-2.714051,-3.046716
2,M251,289,2168.225,['difference'],1,stationary,0.599868,0.628307,0.234093,0.415595,...,0.176457,0.779170,0.840742,0.271204,23.011890,25.267502,13.683699,-44.339754,-42.543653,-32.662704
3,M293,150,5295.457,['detrend'],1,stationary,0.278123,0.306875,0.155947,0.248370,...,0.135658,0.232967,0.258599,0.133951,-19.383345,-18.668373,-16.421203,-11.979653,-7.115953,-14.956206
4,M591,142,372.494,['difference'],1,stationary,0.043832,0.049612,0.021588,0.028322,...,0.014078,0.031077,0.036928,0.015414,-41.046946,-34.349389,-40.052131,-54.762140,-61.631340,-53.341109


## 5. Prepare the forecasting data format

The experimental setup fits forecasting models independently for each series and preserves temporal order. This cell converts the time index to integer format and sorts all observations by series and time.

**Report reference:** Section 3.2 (*Experimental Setup*).

In [5]:
df_series = df.copy()

df_series["ds"] = df_series["ds"].astype("int64")

df_series = df_series.sort_values(
    by=["unique_id", "ds"],
    key=lambda col: col.str.extract(r"(\d+)")[0].astype(int)
              if col.name == "unique_id" else col
).reset_index(drop=True)

df_series

,unique_id,ds,y
0,M80,1,4580.0
1,M80,2,4260.0
2,M80,3,4430.0
3,M80,4,3400.0
4,M80,5,3300.0
...,...,...,...
95748,M47801,75,2364.0
95749,M47801,76,2775.2
95750,M47801,77,1554.2
95751,M47801,78,1096.3


## 6. Train/test split

The final 18 observations of each monthly series are held out as the test horizon, while all earlier observations are used for training.

**Report reference:** Section 3.2 (*Experimental Setup*).

In [6]:
test_size = 18
train_df = df_series.groupby("unique_id").head(-test_size)
test_df = df_series.groupby("unique_id").tail(test_size)

## 7. Lag-based supervised features

Random Forest and LightGBM treat forecasting as a supervised regression problem. The predictors are constructed from past observations and lag transformations: lags `{1, 2, 3, 6, 12}`, rolling means with windows of 3, 6, and 12 observations on lag 1, and an expanding mean on lag 12.

**Report reference:** Section 3.2 (*Models*).

In [7]:
LAGS = [1, 2, 3, 6, 12]
LAG_TRANSFORMS = {
    1:  [(rolling_mean, 3), (rolling_mean, 6), (rolling_mean, 12)],
    12: [(expanding_mean,)],
}

## 8. Per-series hyperparameter tuning with growing-window validation

Tuning Random Forest and LightGBM separately for each individual series. This cell uses `AutoMLForecast` and Optuna with 33 trials per model and series. Validation follows the growing-window setup with `n_windows=6`, `h=18`, and `step_size=1`, and the best parameters are saved for final forecasting.

**Report references:** Section 3.2 (*Models*) and Appendix A, Figure 7.

In [8]:
series_data = {uid: train_df[train_df["unique_id"] == uid].copy() for uid in series_ids}

def fit_auto_mlf_per_series(uid, df_uid, n_windows=6, h=18, num_samples=33):
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    auto_mlf = AutoMLForecast(
        models={
            "rf":   AutoRandomForest(),
            "lgbm": AutoLightGBM(),
        },
        freq=1,
        init_config=lambda trial: {
            "lags": LAGS,
            "lag_transforms": LAG_TRANSFORMS,
            "target_transforms": [Differences([1, 12])],
        },
    )
    auto_mlf.fit(
        df=df_uid,
        n_windows=n_windows,
        h=h,
        step_size=1,
        num_samples=num_samples
    )
    return uid, auto_mlf

results = Parallel(n_jobs=5, verbose=10)(
    delayed(fit_auto_mlf_per_series)(uid, series_data[uid])
    for uid in series_ids
)

auto_mlf_per_series = {uid: auto_mlf for uid, auto_mlf in results}

best_params_rf = {
    uid: amlf.results_["rf"].best_trial.user_attrs["config"]["model_params"]
    for uid, amlf in auto_mlf_per_series.items()
}
best_params_lgbm = {
    uid: amlf.results_["lgbm"].best_trial.user_attrs["config"]["model_params"]
    for uid, amlf in auto_mlf_per_series.items()
}

params_rf_df   = pd.DataFrame([{"unique_id": uid, **p} for uid, p in best_params_rf.items()])
params_lgbm_df = pd.DataFrame([{"unique_id": uid, **p} for uid, p in best_params_lgbm.items()])

params_rf_df.to_csv("params_rf_df.csv", index=False)
params_lgbm_df.to_csv("params_lgbm_df.csv", index=False)

[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done   3 tasks      | elapsed:  1.3min
[Parallel(n_jobs=5)]: Done   8 tasks      | elapsed:  2.5min
[Parallel(n_jobs=5)]: Done  15 tasks      | elapsed:  3.8min
[Parallel(n_jobs=5)]: Done  22 tasks      | elapsed:  5.5min
[Parallel(n_jobs=5)]: Done  31 tasks      | elapsed:  7.7min
[Parallel(n_jobs=5)]: Done  40 tasks      | elapsed:  9.5min
[Parallel(n_jobs=5)]: Done  51 tasks      | elapsed: 11.8min
[Parallel(n_jobs=5)]: Done  62 tasks      | elapsed: 14.4min
[Parallel(n_jobs=5)]: Done  75 tasks      | elapsed: 16.5min
[Parallel(n_jobs=5)]: Done  88 tasks      | elapsed: 18.8min
[Parallel(n_jobs=5)]: Done 103 tasks      | elapsed: 22.2min
[Parallel(n_jobs=5)]: Done 118 tasks      | elapsed: 24.6min
[Parallel(n_jobs=5)]: Done 135 tasks      | elapsed: 28.1min
[Parallel(n_jobs=5)]: Done 152 tasks      | elapsed: 31.4min
[Parallel(n_jobs=5)]: Done 171 tasks      | elapsed: 34.5min
[Parallel(

## 9. Final Random Forest and LightGBM forecasts

After tuning, the best hyperparameters obtained for each series are used to build the final Random Forest and LightGBM models. Each model is fitted to the full training portion of the series and used to forecast the 18-month test horizon.

**Report reference:** Section 3.2 (*Models*).

In [9]:
params_rf_df = pd.read_csv("params_rf_df.csv")
params_lgbm_df = pd.read_csv("params_lgbm_df.csv")

params_rf_dict   = params_rf_df.set_index("unique_id").to_dict(orient="index")
params_lgbm_dict = params_lgbm_df.set_index("unique_id").to_dict(orient="index")

def forecast_series(uid, df_uid, params_rf, params_lgbm, h=18):
    rf_params = params_rf.copy()
    rf_params["n_estimators"]     = int(rf_params["n_estimators"])
    rf_params["max_depth"]        = int(rf_params["max_depth"])
    rf_params["min_samples_split"]= int(rf_params["min_samples_split"])

    lgbm_params = params_lgbm.copy()
    for int_param in ["n_estimators", "max_depth", "num_leaves", "min_child_samples"]:
        if int_param in lgbm_params:
            lgbm_params[int_param] = int(lgbm_params[int_param])

    mlf = MLForecast(
        models={
            "rf":   RandomForestRegressor(**rf_params, random_state=SEED, n_jobs=1),
            "lgbm": LGBMRegressor(**lgbm_params, random_state=SEED, n_jobs=1),
        },
        freq=1,
        lags=LAGS,
        lag_transforms=LAG_TRANSFORMS,
        target_transforms=[Differences([1, 12])],
    )
    mlf.fit(df_uid)
    return mlf.predict(h=h)

results = Parallel(n_jobs=-1, verbose=10)(
    delayed(forecast_series)(uid, series_data[uid], params_rf_dict[uid], params_lgbm_dict[uid])
    for uid in series_ids
)

forecasts_df = pd.concat(results, ignore_index=True)
forecasts_df.to_csv("forecasts_df.csv", index=False)
forecasts_df.head()

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    5.7s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    7.3s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    8.0s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:    8.9s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   10.1s
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:   10.7s
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:   11.9s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:   12.6s
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:   14.0s
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:   15.2s
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:   16.4s
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:   17.7s
[Parallel(n_jobs=-1)]: Done 181 tasks      | elapsed:   18.7s
[Parallel(n_jobs=-1)]: Done 202 tasks      | elapsed:   20.7s
[Parallel(n_jobs=-1)]: Done 225 tasks      | elapsed:  

,unique_id,ds,rf,lgbm
0,M80,568,5871.023316,6236.676819
1,M80,569,4861.062176,5130.924597
2,M80,570,5265.764249,5752.297819
3,M80,571,4362.577720,4920.675786
4,M80,572,4018.575130,4475.624741


## 10. Evaluate Random Forest and LightGBM on the test set

Forecasts are aligned with the held-out test observations and evaluated with NMAE, NRMSE, and SMAPE. The resulting errors are merged into `final_df`, allowing the machine-learning models to be compared with AutoARIMA and the baselines.

**Report reference:** Section 3.2 (*Evaluation Metrics*).

In [10]:
forecasts_eval = forecasts_df.merge(
    test_df[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="left"
)

def nmae(df, models, id_col='unique_id', target_col='y', **kwargs):
    result = {}
    for model in models:
        vals = df.groupby(id_col).apply(
            lambda g: (
                np.mean(np.abs(g[target_col].values - g[model].values)) / np.mean(g[target_col].values)
                if np.mean(g[target_col].values) != 0 else np.nan
            ),
            include_groups=False
        )
        result[model] = vals
    return pd.DataFrame(result).rename_axis(id_col).reset_index()

def nrmse(df, models, id_col='unique_id', target_col='y', **kwargs):
    result = {}
    for model in models:
        vals = df.groupby(id_col).apply(
            lambda g: (
                np.sqrt(np.mean((g[target_col].values - g[model].values) ** 2)) / np.mean(g[target_col].values)
                if np.mean(g[target_col].values) != 0 else np.nan
            ),
            include_groups=False
        )
        result[model] = vals
    return pd.DataFrame(result).rename_axis(id_col).reset_index()

results_eval = evaluate(
    forecasts_eval,
    metrics=[nmae, nrmse, ufl.smape],
    train_df=train_df,
)

for model_name, col_prefix in [("rf", "RF"), ("lgbm", "LGBM")]:
    metrics_df = results_eval.pivot(index='unique_id', columns='metric', values=model_name)
    metrics_df.columns = [f"{col}_{col_prefix}" for col in metrics_df.columns]
    metrics_df = metrics_df.reset_index()

    old_cols = [c for c in final_df.columns if c.endswith(f"_{col_prefix}")]
    final_df = final_df.drop(columns=old_cols, errors='ignore')
    final_df = final_df.merge(metrics_df, on='unique_id', how='left')

final_df.to_csv("final_df.csv", index=False)
final_df.head()

,unique_id,size,complexity,transformations,n_transformations,final_status,nmae_AutoARIMA,nrmse_AutoARIMA,smape_AutoARIMA,nmae_naive,...,%smape_arima_vs_snaive,%nmae_arima_vs_naive,%nrmse_arima_vs_naive,%smape_arima_vs_naive,nmae_RF,nrmse_RF,smape_RF,nmae_LGBM,nrmse_LGBM,smape_LGBM
0,M80,585,22292.093,[],0,stationary,0.119518,0.154520,0.060977,0.323598,...,14.252458,63.065906,56.191667,57.544181,0.176452,0.234333,0.083416,0.238144,0.299127,0.108050
1,M160,91,273.198,['difference'],1,stationary,0.153033,0.178445,0.080407,0.148942,...,-1.173563,-2.746526,-2.714051,-3.046716,0.138339,0.163226,0.071957,0.116905,0.137540,0.059972
2,M251,289,2168.225,['difference'],1,stationary,0.599868,0.628307,0.234093,0.415595,...,13.683699,-44.339754,-42.543653,-32.662704,0.150239,0.234513,0.082709,0.452693,0.512455,0.231991
3,M293,150,5295.457,['detrend'],1,stationary,0.278123,0.306875,0.155947,0.248370,...,-16.421203,-11.979653,-7.115953,-14.956206,0.179351,0.200034,0.082732,0.076821,0.119317,0.038097
4,M591,142,372.494,['difference'],1,stationary,0.043832,0.049612,0.021588,0.028322,...,-40.052131,-54.762140,-61.631340,-53.341109,0.029046,0.036876,0.014900,0.083935,0.100050,0.044980


## 11. Relative improvement against baselines

Naive and Seasonal Naive are reference levels for evaluating whether more complex models are useful. This cell computes the percentage improvement of Random Forest and LightGBM over both baselines for each error metric.

**Report references:** Sections 2.2 (*Forecasting Models*) and 4 (*Results and Analysis*).

In [11]:
def safe_pct(num, den):
    with np.errstate(divide="ignore", invalid="ignore"):
        result = np.where(
            den == 0,
            np.nan,
            ((den - num) / den) * 100
        )
    return result

for col_prefix in ["RF", "LGBM"]:
    p = col_prefix.lower()
    final_df[f"%nmae_{p}_vs_snaive"]  = safe_pct(final_df[f"nmae_{col_prefix}"],  final_df["nmae_seasonal_naive"])
    final_df[f"%nrmse_{p}_vs_snaive"] = safe_pct(final_df[f"nrmse_{col_prefix}"], final_df["nrmse_seasonal_naive"])
    final_df[f"%smape_{p}_vs_snaive"] = safe_pct(final_df[f"smape_{col_prefix}"], final_df["smape_seasonal_naive"])
    final_df[f"%nmae_{p}_vs_naive"]   = safe_pct(final_df[f"nmae_{col_prefix}"],  final_df["nmae_naive"])
    final_df[f"%nrmse_{p}_vs_naive"]  = safe_pct(final_df[f"nrmse_{col_prefix}"], final_df["nrmse_naive"])
    final_df[f"%smape_{p}_vs_naive"]  = safe_pct(final_df[f"smape_{col_prefix}"], final_df["smape_naive"])

pct_cols = [col for col in final_df.columns if col.startswith("%")]
for col in pct_cols:
    final_df[col] = final_df[col].apply(
        lambda x: "Division by Zero" if pd.isna(x) else x
    )

final_df.to_csv("final_df.csv", index=False)
final_df.head()

,unique_id,size,complexity,transformations,n_transformations,final_status,nmae_AutoARIMA,nrmse_AutoARIMA,smape_AutoARIMA,nmae_naive,...,%smape_rf_vs_snaive,%nmae_rf_vs_naive,%nrmse_rf_vs_naive,%smape_rf_vs_naive,%nmae_lgbm_vs_snaive,%nrmse_lgbm_vs_snaive,%smape_lgbm_vs_snaive,%nmae_lgbm_vs_naive,%nrmse_lgbm_vs_naive,%smape_lgbm_vs_naive
0,M80,585,22292.093,[],0,stationary,0.119518,0.154520,0.060977,0.323598,...,-17.301975,45.471753,33.563786,41.920767,-66.014391,-76.275456,-51.943723,26.407359,15.194129,24.768744
1,M160,91,273.198,['difference'],1,stationary,0.153033,0.178445,0.080407,0.148942,...,9.457929,7.118945,6.045789,7.781610,22.745157,22.154531,24.538795,21.509840,20.831313,23.141687
2,M251,289,2168.225,['difference'],1,stationary,0.599868,0.628307,0.234093,0.415595,...,69.502858,63.849701,46.796092,53.127818,41.900545,39.047328,14.458814,-8.926704,-16.260218,-31.471402
3,M293,150,5295.457,['detrend'],1,stationary,0.278123,0.306875,0.155947,0.248370,...,38.237085,27.788533,30.177392,39.014285,67.024745,53.860408,71.559229,69.069742,58.352119,71.917116
4,M591,142,372.494,['difference'],1,stationary,0.043832,0.049612,0.021588,0.028322,...,3.334105,-2.554489,-20.139130,-5.838130,-170.089922,-170.934467,-191.805904,-196.353062,-225.952364,-219.494182
